# Phase 7 — FFN Neuron Specialization: BERT's Emotional Neurons

Each BERT encoder layer contains an FFN intermediate layer of 3072 neurons
(post-GELU). Across 12 layers: 36,864 neurons total. We measure how strongly
each neuron fires for each emotion and connect that signal to SVD truncation.

**Why this matters for compression.** SVD on the FFN intermediate weight matrix
removes directions in the 3072-dim neuron space. If those removed directions
align with the activation patterns of emotion-specific neurons, the
corresponding emotions will be disproportionately damaged.

**Pipeline.**
- **Setup** — paths, imports, helpers.
- **Part A** — extract FFN intermediate activations (post-GELU) via hooks.
- **Part B** — compute Cohen's-d selectivity + catalog top neurons per emotion.
- **Part C** — population analysis (significant counts, distributions).
- **Part D** — SVD-neuron connection + validation against actual F1 drop.
- **Part E** — hierarchical clustering of emotions by shared neuron profiles.
- **Part F** — export everything as CSV to `results/notebook7/`.

## Setup

In [ ]:
import sys, os, pickle, warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    IN_COLAB = False

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, leaves_list
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.compression import (
    apply_svd_compression,
    get_target_layer_names,
    filter_layer_names,
    count_parameters,
)
from src.utils import compute_metrics

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
EXPORT_DIR = os.path.join(RESULTS_DIR, 'notebook7')
os.makedirs(EXPORT_DIR, exist_ok=True)

MODEL_SUBDIR = 'bert-goemotions-23emo-final'
MODEL_PATH = os.path.join(RESULTS_DIR, MODEL_SUBDIR)

EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']

NEURONS_PER_LAYER = 3072
N_LAYERS = 12
BATCH_SIZE = 64

TOP_K = 10
SELECTIVITY_THRESHOLD = 2.0
ANALYSIS_RANK = 128
PCA_COMPONENTS = 50
N_CLUSTERS = 6

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'font.size': 11})

print(f'Project root: {PROJECT_ROOT}')
print(f'Device:       {DEVICE}')
print(f'Export dir:   {EXPORT_DIR}')
print(f'Total FFN neurons: {N_LAYERS * NEURONS_PER_LAYER:,}')

In [ ]:
@torch.no_grad()
def extract_ffn_activations(model, dataset, data_collator, batch_size=BATCH_SIZE):
    """Extract FFN intermediate activations (post-GELU) for every layer."""
    dev = next(model.parameters()).device
    n_layers = model.config.num_hidden_layers

    activations = {l: [] for l in range(n_layers)}
    hooks = []
    for layer_idx in range(n_layers):
        def hook_fn(module, inputs, output, l=layer_idx):
            activations[l].append(output.mean(dim=1).cpu())
        hooks.append(model.bert.encoder.layer[layer_idx]
                     .intermediate.register_forward_hook(hook_fn))

    all_labels = []
    n_batches = (len(dataset) + batch_size - 1) // batch_size
    for i in range(0, len(dataset), batch_size):
        batch = data_collator([dataset[j] for j in range(i, min(i + batch_size, len(dataset)))])
        model(input_ids=batch['input_ids'].to(dev),
              attention_mask=batch['attention_mask'].to(dev))
        all_labels.append(batch['labels'].numpy())
        if (i // batch_size + 1) % 20 == 0:
            print(f'  Batch {i // batch_size + 1}/{n_batches}')

    for h in hooks:
        h.remove()

    for l in range(n_layers):
        activations[l] = torch.cat(activations[l], dim=0).numpy()
    labels = np.concatenate(all_labels, axis=0)
    return activations, labels


def compute_neuron_selectivity(activations, labels):
    """Cohen's-d selectivity per (neuron, emotion) for every layer."""
    n_layers = len(activations)
    n_emotions = labels.shape[1]
    selectivity = {}

    for layer_idx in range(n_layers):
        act = activations[layer_idx]
        sel = np.zeros((act.shape[1], n_emotions))
        for emo_idx in range(n_emotions):
            mask = labels[:, emo_idx] > 0.5
            if mask.sum() < 5 or (~mask).sum() < 5:
                continue
            mean_pos = act[mask].mean(axis=0)
            mean_neg = act[~mask].mean(axis=0)
            std_pos = act[mask].std(axis=0) + 1e-8
            std_neg = act[~mask].std(axis=0) + 1e-8
            n_pos, n_neg = int(mask.sum()), int((~mask).sum())
            pooled = np.sqrt(((n_pos - 1) * std_pos**2 + (n_neg - 1) * std_neg**2)
                             / (n_pos + n_neg - 2)) + 1e-8
            sel[:, emo_idx] = (mean_pos - mean_neg) / pooled
        selectivity[layer_idx] = sel
        print(f'  Layer {layer_idx:2d}: max |selectivity| = {np.abs(sel).max():.3f}, '
              f'mean = {np.abs(sel).mean():.4f}')
    return selectivity


def analyze_svd_neuron_connection(model, selectivity, rank=ANALYSIS_RANK, num_labels=None):
    """Project emotion selectivity onto the kept vs removed FFN SVD subspaces."""
    results = {}
    for layer_idx in range(N_LAYERS):
        W = model.bert.encoder.layer[layer_idx].intermediate.dense.weight.data.float()
        U, S, _ = torch.linalg.svd(W, full_matrices=False)
        U_kept = U[:, :rank].cpu().numpy()
        U_removed = U[:, rank:].cpu().numpy()
        sel = selectivity[layer_idx]

        for emo_idx in range(num_labels):
            neuron_sel = sel[:, emo_idx]
            proj_kept = np.linalg.norm(U_kept.T @ neuron_sel)
            proj_removed = np.linalg.norm(U_removed.T @ neuron_sel)
            total = proj_kept + proj_removed + 1e-8
            results[(layer_idx, emo_idx)] = {
                'frac_in_removed': proj_removed / total,
                'frac_in_kept': proj_kept / total,
            }
        energy = (S[:rank]**2).sum() / (S**2).sum() * 100
        print(f'  Layer {layer_idx:2d}: rank {rank} retains {energy:.1f}% energy '
              f'(sv range [{S[-1].item():.4f}, {S[0].item():.4f}])')
    return results

## Part A — Extract FFN Intermediate Activations

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, problem_type='multi_label_classification',
)
model.eval().to(DEVICE)
params = count_parameters(model)
print(f'Parameters: {params["total"]:,}')

_, _, test_ds, emotion_names, data_collator = load_goemotions(
    exclude_emotions=EXCLUDED_EMOTIONS
)
num_labels = len(emotion_names)
print(f'Test set: {len(test_ds)} examples')
print(f'Emotions ({num_labels}): {emotion_names}')

In [ ]:
print('Extracting FFN intermediate activations from test set...')
activations, labels = extract_ffn_activations(model, test_ds, data_collator)
n_samples = labels.shape[0]
n_layers = len(activations)
n_neurons = activations[0].shape[1]

mem_bytes = sum(activations[l].nbytes for l in range(n_layers)) + labels.nbytes
print(f'\nExtraction complete:')
print(f'  Samples: {n_samples}  |  Layers: {n_layers}  |  Neurons/layer: {n_neurons}')
print(f'  Total memory: {mem_bytes / 1024**2:.1f} MB')
print(f'\nEmotion prevalence in test set:')
for emo_idx in range(num_labels):
    count = int(labels[:, emo_idx].sum())
    print(f'  {emotion_names[emo_idx]:>15s}: {count:>4d} ({count/n_samples*100:5.1f}%)')

## Part B — Neuron Selectivity and Catalog

For each neuron in each layer we compute a standardized effect size (Cohen's d)
for every emotion. |sel| > 2.0 indicates a strong "emotion detector".

In [ ]:
print('Computing neuron selectivity scores...')
selectivity = compute_neuron_selectivity(activations, labels)

all_sel = np.concatenate([selectivity[l].ravel() for l in range(n_layers)])
print(f'\nGlobal selectivity statistics:')
print(f'  Total neuron-emotion pairs: {len(all_sel):,}')
print(f'  Mean |selectivity|: {np.abs(all_sel).mean():.4f}')
print(f'  Max  |selectivity|: {np.abs(all_sel).max():.4f}')
for t in (1.0, 2.0, 3.0):
    print(f'  Pairs with |sel| > {t}: {(np.abs(all_sel) > t).sum():,}')

### B.1 — Catalog of Top Selective Neurons per Emotion

In [ ]:
catalog_rows = []
for emo_idx in range(num_labels):
    scored = []
    for layer_idx in range(n_layers):
        sel = selectivity[layer_idx][:, emo_idx]
        for neuron_idx in range(len(sel)):
            scored.append((layer_idx, neuron_idx, sel[neuron_idx]))
    scored.sort(key=lambda x: abs(x[2]), reverse=True)
    for rank, (layer_idx, neuron_idx, score) in enumerate(scored[:TOP_K]):
        catalog_rows.append({
            'emotion': emotion_names[emo_idx],
            'emotion_idx': emo_idx,
            'rank': rank + 1,
            'layer': layer_idx,
            'neuron': neuron_idx,
            'selectivity': float(score),
            'abs_selectivity': float(abs(score)),
            'direction': 'excitatory' if score > 0 else 'inhibitory',
        })
df_catalog = pd.DataFrame(catalog_rows)

print(f'Neuron catalog: {len(df_catalog)} entries ({num_labels} emotions x {TOP_K} neurons)\n')
print(f'{"Emotion":>15s} | {"#1 (L,N)":>12s} sel   | {"#2 (L,N)":>12s} sel   | {"#3 (L,N)":>12s} sel')
print('-' * 85)
for emo_idx in range(num_labels):
    emo_rows = df_catalog[df_catalog['emotion_idx'] == emo_idx].head(3)
    parts = [f'{emotion_names[emo_idx]:>15s} |']
    for _, row in emo_rows.iterrows():
        parts.append(f' ({row["layer"]:2d},{row["neuron"]:4d}) {row["selectivity"]:+.3f} |')
    print(''.join(parts))

### B.2 — Peak Selectivity Heatmap (Emotion x Layer)

In [ ]:
max_sel_matrix = np.zeros((num_labels, n_layers))
for layer_idx in range(n_layers):
    for emo_idx in range(num_labels):
        max_sel_matrix[emo_idx, layer_idx] = np.abs(selectivity[layer_idx][:, emo_idx]).max()

peak_layer = np.argmax(max_sel_matrix, axis=1)
peak_value = np.max(max_sel_matrix, axis=1)
sort_order = np.lexsort((-peak_value, peak_layer))
sorted_matrix = max_sel_matrix[sort_order]
sorted_names = [emotion_names[i] for i in sort_order]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    sorted_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=[str(l) for l in range(n_layers)],
    yticklabels=sorted_names, ax=ax, linewidths=0.5,
)
ax.set_xlabel('Encoder Layer')
ax.set_ylabel('Emotion')
ax.set_title('Peak Neuron Selectivity by Emotion and Layer\n'
             '(max |selectivity| across 3072 neurons per layer)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'neuron_selectivity_heatmap.png'), bbox_inches='tight')
plt.show()

print('\nEmotions sorted by peak layer:')
for idx in sort_order:
    print(f'  {emotion_names[idx]:>15s}: peak at layer {peak_layer[idx]}, '
          f'max sel = {peak_value[idx]:.3f}')

## Part C — Population Analysis

Count neurons with |selectivity| above threshold per emotion, and examine the
distribution of max-selectivity per neuron across layers.

In [ ]:
sig_counts = []
for emo_idx in range(num_labels):
    per_layer = [int((np.abs(selectivity[l][:, emo_idx]) > SELECTIVITY_THRESHOLD).sum())
                 for l in range(n_layers)]
    sig_counts.append({
        'emotion': emotion_names[emo_idx],
        'total_significant': sum(per_layer),
        **{f'layer_{l}': per_layer[l] for l in range(n_layers)},
    })
df_sig = pd.DataFrame(sig_counts).sort_values('total_significant', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(df_sig)))
ax.barh(range(len(df_sig)), df_sig['total_significant'], color=colors)
ax.set_yticks(range(len(df_sig)))
ax.set_yticklabels(df_sig['emotion'])
ax.set_xlabel(f'Specialized Neurons (|selectivity| > {SELECTIVITY_THRESHOLD})')
ax.set_title(f'Neuron Specialization by Emotion\n'
             f'({n_layers} layers x {n_neurons} neurons = {n_layers * n_neurons:,} total)',
             fontsize=13, fontweight='bold')
for i, val in enumerate(df_sig['total_significant']):
    ax.text(val + 5, i, str(int(val)), va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'neuron_specialization_per_emotion.png'), bbox_inches='tight')
plt.show()

print('\nFewest specialized neurons (most fragile under compression):')
for _, row in df_sig.head(5).iterrows():
    print(f'  {row["emotion"]:>15s}: {int(row["total_significant"]):4d} neurons')
print('\nMost specialized neurons:')
for _, row in df_sig.tail(5).iterrows():
    print(f'  {row["emotion"]:>15s}: {int(row["total_significant"]):4d} neurons')

### C.1 — Distribution of Max |selectivity| per Neuron, by Layer

In [ ]:
per_layer_stats = []
fig, axes = plt.subplots(3, 4, figsize=(18, 12), sharey=True)
for layer_idx in range(n_layers):
    ax = axes[layer_idx // 4, layer_idx % 4]
    sel = selectivity[layer_idx]
    max_sel_per_neuron = np.abs(sel).max(axis=1)

    ax.hist(max_sel_per_neuron, bins=60, color='steelblue',
            alpha=0.7, edgecolor='white')
    ax.axvline(x=SELECTIVITY_THRESHOLD, color='red', linestyle='--',
               linewidth=1.5, label=f'threshold = {SELECTIVITY_THRESHOLD}')
    n_spec = int((max_sel_per_neuron > SELECTIVITY_THRESHOLD).sum())
    pct = n_spec / len(max_sel_per_neuron) * 100
    ax.set_title(f'Layer {layer_idx}\n{n_spec} specialized ({pct:.1f}%)',
                 fontsize=10, fontweight='bold')
    if layer_idx >= 8:
        ax.set_xlabel('Max |selectivity|')
    if layer_idx % 4 == 0:
        ax.set_ylabel('Neuron count')
    per_layer_stats.append({
        'layer': layer_idx,
        'n_specialized': n_spec,
        'fraction_specialized': pct / 100,
        'mean_max_selectivity': float(max_sel_per_neuron.mean()),
        'median_max_selectivity': float(np.median(max_sel_per_neuron)),
        'max_max_selectivity': float(max_sel_per_neuron.max()),
    })
axes[0, 0].legend(fontsize=8)
fig.suptitle('Distribution of Neuron Specialization per Layer\n'
             '(max |selectivity| across emotions for each neuron)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'neuron_selectivity_distributions.png'), bbox_inches='tight')
plt.show()

df_per_layer = pd.DataFrame(per_layer_stats)
print('\nSpecialized neurons per layer:')
for row in per_layer_stats:
    print(f'  Layer {row["layer"]:2d}: {row["n_specialized"]:4d} / {n_neurons} '
          f'({row["fraction_specialized"]*100:.1f}%)')

## Part D — SVD-Neuron Connection

For each FFN intermediate weight $W = U \Sigma V^{T}$, the first $k$ columns of
$U$ span the subspace SVD retains; the rest is discarded. We project the
emotion selectivity vector onto both subspaces — if most of the signal lives
in the removed subspace, SVD will hurt that emotion.

In [ ]:
print(f'Analyzing SVD-neuron connection at rank {ANALYSIS_RANK}...')
svd_neuron_results = analyze_svd_neuron_connection(model, selectivity,
                                                    rank=ANALYSIS_RANK,
                                                    num_labels=num_labels)

frac_removed_matrix = np.zeros((num_labels, n_layers))
for layer_idx in range(n_layers):
    for emo_idx in range(num_labels):
        frac_removed_matrix[emo_idx, layer_idx] = \
            svd_neuron_results[(layer_idx, emo_idx)]['frac_in_removed']

mean_frac_removed = frac_removed_matrix.mean(axis=1)
sort_order_vuln = np.argsort(-mean_frac_removed)
sorted_frac = frac_removed_matrix[sort_order_vuln]
sorted_names_vuln = [emotion_names[i] for i in sort_order_vuln]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    sorted_frac, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=[str(l) for l in range(n_layers)],
    yticklabels=sorted_names_vuln, ax=ax, linewidths=0.5,
    vmin=0, vmax=0.7,
)
ax.set_xlabel('Encoder Layer')
ax.set_ylabel('Emotion (sorted by vulnerability)')
ax.set_title(f'Fraction of Emotion Signal in SVD-Removed Subspace (rank={ANALYSIS_RANK})\n'
             'Higher = more signal destroyed by SVD truncation',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'svd_neuron_vulnerability_heatmap.png'), bbox_inches='tight')
plt.show()

print('\nMost vulnerable emotions (highest frac in removed subspace):')
for idx in sort_order_vuln[:5]:
    print(f'  {emotion_names[idx]:>15s}: {mean_frac_removed[idx]:.3f}')
print('\nLeast vulnerable emotions:')
for idx in sort_order_vuln[-5:]:
    print(f'  {emotion_names[idx]:>15s}: {mean_frac_removed[idx]:.3f}')

### D.1 — Validation: Predicted Vulnerability vs Actual F1 Drop

In [ ]:
print(f'Compressing FFN intermediate at rank {ANALYSIS_RANK} for validation...')
all_layer_names = get_target_layer_names(model)
ffn_inter_names = filter_layer_names(all_layer_names, component='intermediate')
print(f'  Compressing {len(ffn_inter_names)} layers')

compressed_model = apply_svd_compression(model, rank=ANALYSIS_RANK, layer_names=ffn_inter_names)
compressed_model.to(DEVICE)

eval_args = TrainingArguments(
    output_dir=os.path.join(RESULTS_DIR, 'tmp_eval'),
    per_device_eval_batch_size=BATCH_SIZE,
    fp16=(DEVICE == 'cuda'),
    report_to='none',
)
trainer_base = Trainer(model=model, args=eval_args,
                       compute_metrics=compute_metrics, data_collator=data_collator)
trainer_comp = Trainer(model=compressed_model, args=eval_args,
                       compute_metrics=compute_metrics, data_collator=data_collator)

print('  Evaluating baseline...')
base_metrics = trainer_base.evaluate(test_ds)
print('  Evaluating compressed...')
comp_metrics = trainer_comp.evaluate(test_ds)
print(f'  Baseline   F1 macro: {base_metrics["eval_f1_macro"]:.4f}')
print(f'  Compressed F1 macro: {comp_metrics["eval_f1_macro"]:.4f}')

baseline_f1_per_emotion = np.array([base_metrics.get(f'eval_f1_label_{i}', np.nan)
                                    for i in range(num_labels)])
compressed_f1_per_emotion = np.array([comp_metrics.get(f'eval_f1_label_{i}', np.nan)
                                      for i in range(num_labels)])
per_emotion_f1_drop = baseline_f1_per_emotion - compressed_f1_per_emotion

del compressed_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
valid_mask = ~np.isnan(per_emotion_f1_drop)
x = mean_frac_removed[valid_mask]
y = per_emotion_f1_drop[valid_mask]
valid_names = [emotion_names[i] for i in range(num_labels) if valid_mask[i]]
rho, p_value = spearmanr(x, y)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(x, y, s=80, c='steelblue', edgecolor='white', linewidth=0.5, zorder=3)
for i, name in enumerate(valid_names):
    ax.annotate(name, (x[i], y[i]), fontsize=7, ha='center', va='bottom',
                xytext=(0, 5), textcoords='offset points')
z = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, np.polyval(z, x_line), '--', color='red', alpha=0.7, linewidth=1.5)
ax.set_xlabel('Fraction of Signal in Removed Subspace (SVD vulnerability)')
ax.set_ylabel(f'F1 Drop under Compression (rank={ANALYSIS_RANK})')
ax.set_title(f'SVD Vulnerability vs Actual F1 Impact\n'
             f'Spearman rho = {rho:.3f}, p = {p_value:.4f}',
             fontsize=13, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'svd_vulnerability_vs_f1_drop.png'), bbox_inches='tight')
plt.show()

print(f'Spearman correlation: rho = {rho:.3f}, p = {p_value:.4f}')
if p_value < 0.05:
    print('=> Significant correlation: emotions whose neuron signal lies in the')
    print('   SVD-removed subspace do suffer more F1 degradation.')
else:
    print('=> Correlation not significant at p < 0.05 — relationship may be more complex.')

## Part E — Emotion Clustering by Shared Neuron Profiles

Stack each emotion's selectivity vector across all layers (12 x 3072 = 36,864
features), reduce to 50 PCs, then hierarchical cluster. Emotions that share
specialised neurons should end up in the same subtree.

In [ ]:
emotion_features = np.zeros((num_labels, n_layers * n_neurons))
for emo_idx in range(num_labels):
    for layer_idx in range(n_layers):
        start = layer_idx * n_neurons
        emotion_features[emo_idx, start:start + n_neurons] = selectivity[layer_idx][:, emo_idx]
print(f'Emotion feature matrix: {emotion_features.shape}')

pca = PCA(n_components=PCA_COMPONENTS, random_state=42)
emotion_features_pca = pca.fit_transform(emotion_features)
explained = pca.explained_variance_ratio_.sum()
print(f'PCA: {PCA_COMPONENTS} components explain {explained*100:.1f}% of variance')

distances = pdist(emotion_features_pca, metric='cosine')
linkage_matrix = linkage(distances, method='ward')

fig, ax = plt.subplots(figsize=(14, 8))
dendrogram(linkage_matrix, labels=emotion_names,
           leaf_rotation=45, leaf_font_size=10, ax=ax,
           color_threshold=0.7 * linkage_matrix[-1, 2])
ax.set_ylabel('Distance')
ax.set_title('Emotional Genealogy: Hierarchical Clustering of Neuron Profiles\n'
             '(emotions sharing specialized neurons cluster together)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'neuron_emotion_dendrogram.png'), bbox_inches='tight')
plt.show()

cluster_labels_arr = fcluster(linkage_matrix, N_CLUSTERS, criterion='maxclust')
print(f'\nCluster assignments ({N_CLUSTERS} clusters):')
for c in range(1, N_CLUSTERS + 1):
    members = [emotion_names[i] for i in range(num_labels) if cluster_labels_arr[i] == c]
    print(f'  Cluster {c}: {members}')

### E.1 — Emotion Similarity Matrix (Dendrogram-ordered)

In [ ]:
sim_matrix = 1 - squareform(distances)
leaf_order = leaves_list(linkage_matrix)
sim_reordered = sim_matrix[np.ix_(leaf_order, leaf_order)]
names_reordered = [emotion_names[i] for i in leaf_order]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(sim_reordered, cmap='RdBu_r',
            xticklabels=names_reordered, yticklabels=names_reordered,
            ax=ax, linewidths=0.3, vmin=-0.5, vmax=1.0, square=True)
ax.set_title('Emotion Similarity via Shared Neuron Selectivity Profiles\n'
             '(cosine similarity in PCA-reduced space, ordered by dendrogram)',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'neuron_emotion_similarity.png'), bbox_inches='tight')
plt.show()

## Part F — Export

All results as CSV under `results/notebook7/` (plus two legacy .pkl files):

- `neuron_catalog.csv` — top K neurons per emotion.
- `neuron_significant_counts.csv` — per-emotion significant-neuron counts.
- `neuron_emotion_clusters.csv` — cluster assignments + vulnerability + n_sig.
- `peak_selectivity_matrix.csv` — max |selectivity| per (emotion, layer).
- `svd_vulnerability_matrix.csv` — frac-in-removed per (emotion, layer).
- `svd_vulnerability_summary.csv` — emotion x mean_frac_removed + F1 drop.
- `per_layer_specialization_stats.csv` — distribution stats per layer.
- `emotion_similarity_matrix.csv` — 23 x 23 cosine similarity.
- `neuron_analysis_summary.csv` — one-row master summary.

And `.pkl` artefacts for downstream notebooks:
- `neuron_selectivity.pkl`     — full (12 x 3072 x 23) selectivity dict.
- `svd_neuron_connection.pkl` — SVD projection results.

In [ ]:
exports = {}

# 1. Catalog
exports['neuron_catalog.csv'] = df_catalog

# 2. Significant counts
exports['neuron_significant_counts.csv'] = df_sig

# 3. Peak selectivity (wide)
peak_df = pd.DataFrame(max_sel_matrix,
                       index=emotion_names,
                       columns=[f'layer_{l}' for l in range(n_layers)])
peak_df.index.name = 'emotion'
peak_df['peak_layer'] = peak_layer
peak_df['peak_value'] = peak_value
exports['peak_selectivity_matrix.csv'] = peak_df.reset_index()

# 4. SVD vulnerability (wide)
vuln_df = pd.DataFrame(frac_removed_matrix,
                       index=emotion_names,
                       columns=[f'layer_{l}' for l in range(n_layers)])
vuln_df.index.name = 'emotion'
vuln_df['mean_frac_removed'] = mean_frac_removed
exports['svd_vulnerability_matrix.csv'] = vuln_df.reset_index()

# 5. Vulnerability summary
exports['svd_vulnerability_summary.csv'] = pd.DataFrame({
    'emotion': emotion_names,
    'mean_frac_removed': mean_frac_removed,
    'baseline_f1': baseline_f1_per_emotion,
    'compressed_f1': compressed_f1_per_emotion,
    'f1_drop': per_emotion_f1_drop,
})

# 6. Per-layer specialization stats
exports['per_layer_specialization_stats.csv'] = df_per_layer

# 7. Similarity matrix (ordered by dendrogram)
sim_df = pd.DataFrame(sim_reordered, index=names_reordered, columns=names_reordered)
sim_df.index.name = 'emotion'
exports['emotion_similarity_matrix.csv'] = sim_df.reset_index()

# 8. Cluster assignments
exports['neuron_emotion_clusters.csv'] = pd.DataFrame({
    'emotion': emotion_names,
    'cluster': cluster_labels_arr,
    'mean_frac_removed': mean_frac_removed,
    'total_sig_neurons': [int(df_sig[df_sig['emotion'] == e]['total_significant'].values[0])
                          for e in emotion_names],
})

# 9. Master summary
total_sig_neurons = sum(
    int((np.abs(selectivity[l]).max(axis=1) > SELECTIVITY_THRESHOLD).sum())
    for l in range(n_layers)
)
exports['neuron_analysis_summary.csv'] = pd.DataFrame([{
    'n_samples': n_samples,
    'n_layers': n_layers,
    'n_neurons_per_layer': n_neurons,
    'n_neurons_total': n_layers * n_neurons,
    'n_emotions': num_labels,
    'selectivity_threshold': SELECTIVITY_THRESHOLD,
    'analysis_rank': ANALYSIS_RANK,
    'n_clusters': N_CLUSTERS,
    'total_specialized_neurons': total_sig_neurons,
    'fraction_specialized': total_sig_neurons / (n_layers * n_neurons),
    'max_selectivity_observed': float(np.abs(all_sel).max()),
    'mean_selectivity_abs': float(np.abs(all_sel).mean()),
    'baseline_f1_macro': float(base_metrics['eval_f1_macro']),
    'compressed_f1_macro': float(comp_metrics['eval_f1_macro']),
    'spearman_vulnerability_vs_f1drop': rho,
    'spearman_p_value': p_value,
    'pca_variance_explained': float(explained),
}])

for fname, df in exports.items():
    out_path = os.path.join(EXPORT_DIR, fname)
    df.to_csv(out_path, index=False)
    print(f'  [{len(df):>5d} rows] {fname}')

# Legacy pickles
with open(os.path.join(EXPORT_DIR, 'neuron_selectivity.pkl'), 'wb') as f:
    pickle.dump(selectivity, f)
with open(os.path.join(EXPORT_DIR, 'svd_neuron_connection.pkl'), 'wb') as f:
    pickle.dump({
        'rank': ANALYSIS_RANK,
        'results': svd_neuron_results,
        'frac_removed_matrix': frac_removed_matrix,
        'mean_frac_removed': mean_frac_removed,
        'per_emotion_f1_drop': per_emotion_f1_drop,
    }, f)
print('  [pickled]  neuron_selectivity.pkl')
print('  [pickled]  svd_neuron_connection.pkl')

print('\n' + '=' * 70)
print('SUMMARY - Phase 7: FFN Neuron Specialization Analysis')
print('=' * 70)
print(f'Neurons analysed: {n_layers} x {n_neurons} = {n_layers * n_neurons:,}')
print(f'Specialized ({SELECTIVITY_THRESHOLD}+): {total_sig_neurons:,} '
      f'({total_sig_neurons / (n_layers * n_neurons) * 100:.1f}%)')
print(f'Max selectivity: {np.abs(all_sel).max():.3f}')
print(f'SVD-neuron correlation: Spearman rho = {rho:.3f} (p = {p_value:.4f})')
print(f'\nExported {len(exports)} CSVs (+2 .pkl) to {EXPORT_DIR}')